

# 逐一详细回答

---

## 一、IQ采样数据 vs 原始时域波形

```
本质区别：

IQ采样（复数基带信号）:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
x[n] = I[n] + j·Q[n]

                    天线接收
                       │
                       ▼
              ┌─────────────────┐
              │  射频信号 (RF)   │  f_carrier = 例如 10 MHz
              │  s(t) = A(t)·cos(2πf_c·t + φ(t))
              └────────┬────────┘
                       │ 下变频 (混频器 × 本振)
                       │
            ┌──────────┴──────────┐
            ▼                     ▼
    ┌──────────────┐      ┌──────────────┐
    │ × cos(2πf_c·t)│      │ × sin(2πf_c·t)│
    │   ↓ 低通滤波  │      │   ↓ 低通滤波  │
    │   I(t) 同相   │      │   Q(t) 正交   │
    └──────┬───────┘      └──────┬───────┘
           │                      │
           │      ADC采样          │      ADC采样
           ▼                      ▼
         I[n]                   Q[n]
           │                      │
           └──────────┬───────────┘
                      ▼
              x[n] = I[n] + j·Q[n]   ← IQ采样
              shape: [2, N] 或 复数数组 [N]


原始时域波形（实数信号）:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
              ┌─────────────────┐
              │  射频/中频信号   │
              │  s(t) = A(t)·cos(2πf_c·t + φ(t))
              └────────┬────────┘
                       │ 直接ADC采样（不下变频，或仅到中频）
                       ▼
                     s[n]   ← 实数时域波形
                     shape: [1, N]
```

```
关键差异对比：
┌──────────────┬────────────────────────┬──────────────────────┐
│ 特性          │ IQ采样 (复基带)         │ 原始时域波形 (实信号)  │
├──────────────┼────────────────────────┼──────────────────────┤
│ 数据类型      │ 复数 (I + jQ)          │ 实数                  │
│ 通道数        │ 2 (I路 + Q路)          │ 1                    │
│ 载频信息      │ 已被去除(下变频到0Hz)   │ 保留在信号中           │
│ 采样率需求    │ ≥ 2×信号带宽           │ ≥ 2×最高频率          │
│              │ (如: 8kHz即可)          │ (如: 需要20MHz+)      │
│ 信息完整性    │ 保留幅度+相位           │ 保留幅度,相位隐含      │
│ 数据量        │ 2N (I和Q各N点)         │ N (但N可能很大)       │
│ 频谱          │ 以0Hz为中心, 双边       │ 以载频为中心           │
│ 后续处理      │ 直接分析调制特征        │ 需先下变频到基带       │
│ 深度学习输入  │ [2, N] 双通道 ⭐推荐    │ [1, N] 单通道         │
└──────────────┴────────────────────────┴──────────────────────┘

结论：
━━━━
对于Link11识别项目, 强烈推荐使用 IQ采样数据
原因:
  1. IQ已去除载频, 专注于调制信息(这正是区分协议的关键)
  2. 采样率低(8kHz vs 20MHz+), 数据量小得多
  3. 保留完整的幅度和相位信息
  4. 深度学习双通道输入, 天然适合CNN处理
  5. 所有有意义的特征(频谱、星座图、相位变化)都可从IQ直接计算

如果原始采集的是实数中频/射频信号:
  → 预处理第一步就应该数字下变频转为IQ基带
```

---

## 二、元信息（Metadata）完整列表及使用方式

```
元信息完整清单：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

┌─ 采集设备参数 ─────────────────────────────────────────┐
│                                                         │
│  sample_rate:        采样率 (Hz)         例: 8000       │
│  center_frequency:   中心频率 (Hz)       例: 10.235e6   │
│  bandwidth:          接收带宽 (Hz)       例: 4000       │
│  adc_bits:           ADC位数             例: 16         │
│  antenna_type:       天线类型            例: "whip"     │
│  receiver_model:     接收机型号          例: "SDR-XX"   │
│  gain_db:            接收增益 (dB)       例: 30         │
│  agc_enabled:        AGC是否开启         例: True       │
│                                                         │
├─ 信号质量参数 ─────────────────────────────────────────┤
│                                                         │
│  estimated_snr_db:   估计信噪比 (dB)     例: 15.3      │
│  noise_floor_dbm:    底噪 (dBm)         例: -110       │
│  signal_power_dbm:   信号功率 (dBm)      例: -85        │
│  has_interference:   是否有干扰          例: False      │
│  clipping_ratio:     削波比例            例: 0.001      │
│                                                         │
├─ 时间与空间参数 ───────────────────────────────────────┤
│                                                         │
│  timestamp:          采集时间戳          例: 1700000000 │
│  duration_sec:       信号时长 (秒)       例: 1.024      │
│  num_samples:        采样点数            例: 8192       │
│  geographic_location:采集地点(经纬度)    例: [39.9,116.4]│
│  propagation_mode:   传播方式            例: "skywave"  │
│                                                         │
├─ 标注信息 ─────────────────────────────────────────────┤
│                                                         │
│  label:              标签               例: "link11"    │
│  sub_label:          子标签             例: "clew"      │
│  confidence:         标注置信度          例: 0.95       │
│  annotator:          标注者             例: "expert_1"  │
│  link11_mode:        Link11模式          例: "A"        │
│  data_rate_bps:      数据速率            例: 1364       │
│                                                         │
├─ 预处理状态 ───────────────────────────────────────────┤
│                                                         │
│  is_baseband:        是否已下变频到基带   例: True      │
│  is_normalized:      是否已功率归一化     例: False     │
│  original_sample_rate: 原始采样率        例: 48000      │
│  filter_applied:     已应用的滤波器      例: "BPF 300-3400"│
│                                                         │
└─────────────────────────────────────────────────────────┘
```

```
元信息使用方式：
━━━━━━━━━━━━━━━━━

1. 预处理阶段使用：
   ┌─────────────────────────────────────────────────┐
   │ sample_rate → 决定是否需要重采样               │
   │ center_frequency → 数字下变频参数              │
   │ bandwidth → 带通滤波器设计                     │
   │ is_baseband → 决定是否跳过下变频步骤           │
   │ adc_bits → 量化噪声估计                       │
   └─────────────────────────────────────────────────┘

2. 数据筛选/分组：
   ┌─────────────────────────────────────────────────┐
   │ estimated_snr_db → 按SNR分组训练/评估          │
   │   (分析模型在不同SNR下的性能曲线)              │
   │ has_interference → 分离干净/有干扰样本         │
   │ propagation_mode → 区分天波/地波场景           │
   │ link11_mode → 区分Link11 A/B模式              │
   └─────────────────────────────────────────────────┘

3. 模型输入条件化（可选高级用法）：
   ┌─────────────────────────────────────────────────┐
   │ estimated_snr_db → 作为模型的条件输入           │
   │   (Conditional模型, 让模型知道当前SNR)          │
   │ sample_rate → 如果训练数据采样率不统一,         │
   │   模型需要知道时间/频率尺度                     │
   └─────────────────────────────────────────────────┘

4. 评估分析：
   ┌─────────────────────────────────────────────────┐
   │ 按SNR绘制检测概率曲线 Pd vs SNR               │
   │ 按传播条件分析性能差异                         │
   │ 按时间分析模型是否有时变退化                   │
   │ 分析不同接收设备对性能的影响                   │
   └─────────────────────────────────────────────────┘

5. 质量控制：
   ┌─────────────────────────────────────────────────┐
   │ clipping_ratio > 0.05 → 丢弃(削波严重)        │
   │ confidence < 0.8 → 标注不可信, 谨慎使用        │
   │ num_samples < 最小要求 → 丢弃                  │
   └─────────────────────────────────────────────────┘
```

```python
# metadata.csv 示例
"""
file_path, label, sample_rate, center_freq, snr_db, duration, num_samples, 
link11_mode, data_rate, is_baseband, propagation, timestamp, confidence

train/link11/sample_0001.npy, link11, 8000, 10.235e6, 15.3, 1.024, 8192, A, 1364, True, skywave, 1700000000, 0.98
train/link11/sample_0002.npy, link11, 8000, 8.421e6,  8.7,  1.024, 8192, B, 2250, True, groundwave, 1700003600, 0.95
train/non_link11/noise_0001.npy, noise, 8000, 10.0e6, -inf, 1.024, 8192, -, -, True, -, 1700007200, 1.0
train/non_link11/ale_0001.npy, ale, 8000, 12.5e6, 12.0, 1.024, 8192, -, -, True, skywave, 1700010800, 0.90
"""
```

---

## 三、单音模式的考虑

```
Link11信号完整模式：
━━━━━━━━━━━━━━━━━━━━

实际上Link11包含多种信号段, 不仅仅是多音:

┌─────── Link11 完整传输帧结构 ──────────────────────────┐
│                                                         │
│  ┌──────┐  ┌──────────┐  ┌──────────────┐  ┌───────┐ │
│  │ CLEW │→│  DATA     │→│   DATA       │→│ EOM   │ │
│  │ 呼叫 │  │  (多音    │  │  (继续)      │  │ 结束  │ │
│  │ 信号 │  │   DPSK)   │  │              │  │ 标记  │ │
│  └──┬───┘  └────┬─────┘  └──────┬───────┘  └───┬───┘ │
│     │           │               │               │      │
│     ▼           ▼               ▼               ▼      │
│  15音调       15音调并行      15音调并行      特定     │
│  并行发送     DPSK调制        DPSK调制        模式     │
│  (固定频率)   (数据承载)      (数据承载)              │
│                                                         │
│  ← 全程都是多音并行调制 →                              │
│                                                         │
└─────────────────────────────────────────────────────────┘

但实际截获中可能遇到的"单音"场景：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

场景1: 部分截获
┌─────────────────────────────────────┐
│ 实际截获的信号段可能只包含:         │
│ · 只截到帧的一小段 → 部分音调      │
│ · 只截到单个子载波 → 看起来像单音  │
│ · 带通滤波后只保留一个音 → 单音    │
└─────────────────────────────────────┘

场景2: 信道衰落
┌─────────────────────────────────────┐
│ HF信道频率选择性衰落:              │
│ · 部分频率的音调被深衰落淹没       │
│ · 可能只剩几个或一个强音调         │
│ · 特别是窄带干扰遮蔽部分音调       │
└─────────────────────────────────────┘

场景3: 低SNR
┌─────────────────────────────────────┐
│ 信噪比很低时:                      │
│ · 只有最强的一两个音调可见          │
│ · 其他音调淹没在噪声中             │
│ · 频谱上可能只呈现单音特征          │
└─────────────────────────────────────┘

因此, 系统必须考虑以下所有情况:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

应该纳入的信号形态：
┌────────────────────────────────────────────────┐
│ 类型              │ 可见音调数 │ 训练中应包含  │
├───────────────────┼───────────┼───────────────┤
│ 完整CLEW          │ 15        │ ✅ 大量       │
│ 完整DATA段        │ 15        │ ✅ 大量       │
│ 衰落后部分音调    │ 5~14      │ ✅ 增强生成   │
│ 严重衰落/低SNR    │ 1~4       │ ✅ 增强生成   │
│ 单音调(极端退化)  │ 1         │ ✅ 增强生成   │
│ CLEW+DATA过渡段   │ 混合      │ ✅ 包含       │
│ EOM段             │ 特定      │ ✅ 包含       │
└────────────────────────────────────────────────┘

数据增强中模拟退化：
┌────────────────────────────────────────────────┐
│ def simulate_tone_loss(signal, fs, n_drop):    │
│     """模拟部分音调丢失"""                      │
│     # 随机选择n_drop个音调频率                  │
│     # 在这些频率处加深度陷波                    │
│     # 模拟频率选择性衰落效果                    │
│                                                 │
│ def simulate_partial_capture(signal, ratio):    │
│     """模拟部分截获"""                          │
│     # 只取信号的一部分                          │
│     # 或只保留部分带宽                          │
│     start = random.randint(...)                 │
│     length = int(len(signal) * ratio)           │
│     return signal[start:start+length]           │
│                                                 │
│ def simulate_narrowband_filter(signal, fs, bw): │
│     """模拟窄带滤波(只留少数音调)"""            │
│     # 模拟接收机带宽不足的情况                  │
│     # bw < 3kHz → 丢失边缘音调                 │
└────────────────────────────────────────────────┘

对模型的影响：
━━━━━━━━━━━━
模型不能仅依赖"检测到15个等间隔音调"这单一判据,
还需要学习:
  · 单个或少数Link11音调的特征(DPSK调制特性)
  · 音调频率的绝对位置(605Hz的整数倍)
  · 单音调的DPSK相位变化规律
  · 即使只有部分音调, 其间距仍是605Hz的倍数

→ 这正是深度学习方法的价值所在:
  传统多音检测在退化条件下失效,
  深度学习可以从更底层特征中学习判别
```

---

## 四、多视图输入组合策略

```
核心思想: 不同的输入表示从不同物理角度描述信号,
         每个角度提供独立的、互补的判别信息

多视图输入设计：
━━━━━━━━━━━━━━━

┌─────────────────────────────────────────────────────────────┐
│                                                              │
│  View 1: IQ时域 [2, N]                                      │
│  ──────────────────────                                      │
│  物理角度: 信号的瞬时幅度和相位随时间的变化                    │
│  捕获特征:                                                   │
│   · DPSK相位跳变模式 (符号间相位差)                          │
│   · 包络波动特征 (多音叠加的包络)                            │
│   · 时域周期性 (符号速率相关)                                │
│   · 瞬态特征 (CLEW→DATA过渡)                                │
│  可靠角度: 符号级调制特征, 时序依赖关系                      │
│  独特信息: 相位连续性、符号间转换 (其他视图无法直接提供)      │
│                                                              │
│  View 2: 频谱/PSD [1, N_fft]                                │
│  ──────────────────────                                      │
│  物理角度: 信号能量在频率上的分布                             │
│  捕获特征:                                                   │
│   · 多音调的梳状谱结构 ⭐ (Link11最显著特征)                 │
│   · 音调间隔 (~605Hz)                                        │
│   · 各音调的相对幅度                                         │
│   · 信号带宽                                                 │
│  可靠角度: 频域结构特征, 对时移不敏感                        │
│  独特信息: 频率间隔的精确度 (时域不直观)                     │
│                                                              │
│  View 3: STFT时频图 [1, T, F] 或 [2, T, F]                 │
│  ──────────────────────                                      │
│  物理角度: 信号频率成分随时间的演变                           │
│  捕获特征:                                                   │
│   · 各音调的时间连续性 (在时频图上是水平线)                  │
│   · 信号起止、帧结构                                        │
│   · 频率随时间的稳定性                                       │
│   · CLEW与DATA段的过渡模式                                   │
│  可靠角度: 时频联合特征, 兼顾时域和频域                      │
│  独特信息: 频率的时间稳定性 (静态频谱无法反映)               │
│                                                              │
│  View 4: 自相关函数 (ACF) [1, N_lag]                        │
│  ──────────────────────                                      │
│  物理角度: 信号的自相似性/周期性                              │
│  捕获特征:                                                   │
│   · 符号周期 (1/1364≈0.733ms处的峰值)                       │
│   · 多音调导致的自相关周期结构                               │
│   · 信号的带宽 (ACF主瓣宽度)                                │
│  可靠角度: 周期性特征, 对SNR有一定鲁棒性                    │
│  独特信息: 符号速率的直接体现                                │
│                                                              │
│  View 5: 瞬时特征序列 [3, N] 或 [4, N]                     │
│  ──────────────────────                                      │
│  物理角度: 信号的瞬时统计量随时间变化                         │
│  包含:                                                       │
│   · 瞬时幅度 A[n] = |x[n]|                                  │
│   · 瞬时相位 φ[n] = angle(x[n])                             │
│   · 瞬时频率 f[n] = dφ/dt                                   │
│   · 瞬时相位差 Δφ[n] = φ[n]-φ[n-1] (DPSK关键!)            │
│  可靠角度: 调制方式识别的经典特征                            │
│  独特信息: 瞬时频率稳定性, DPSK差分相位                      │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

```
推荐的多视图组合方案：
━━━━━━━━━━━━━━━━━━━━

方案A: 三视图（推荐, 平衡性能与复杂度）⭐⭐⭐
┌──────────────────────────────────────────────────┐
│  Input 1: IQ时域       [2, 8192]    → 1D-CNN     │
│  Input 2: PSD频谱      [1, 4096]    → 1D-CNN     │
│  Input 3: STFT时频图   [1, 128, 65] → 2D-CNN     │
│                                                    │
│  互补性分析:                                       │
│  · IQ: 捕获时序/调制细节 (时域)                   │
│  · PSD: 捕获频谱结构   (频域)                     │
│  · STFT: 捕获时频演变  (时频联合)                 │
│  → 三者覆盖了信号的三个正交描述维度               │
└──────────────────────────────────────────────────┘

方案B: 四视图（更全面）
┌──────────────────────────────────────────────────┐
│  Input 1: IQ时域       [2, 8192]    → 1D-CNN     │
│  Input 2: PSD频谱      [1, 4096]    → 1D-CNN     │
│  Input 3: STFT时频图   [1, 128, 65] → 2D-CNN     │
│  Input 4: ACF自相关    [1, 2048]    → 1D-CNN     │
│                                                    │
│  增加ACF的价值:                                    │
│  · 直接反映周期性, 对符号速率敏感                 │
│  · 与PSD是傅里叶变换对, 但数值特性不同            │
│  · 对噪声的鲁棒性优于直接频谱                     │
└──────────────────────────────────────────────────┘

为什么这些视图具有独立可靠的角度？
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   数学关系    │    信息侧重     │   失效条件不同
───────────────┼─────────────────┼──────────────────
IQ ←FFT→ 频谱  │ IQ: 局部细节    │ IQ: 对频偏敏感
               │ 频谱: 全局结构  │ 频谱: 对时移免疫
───────────────┼─────────────────┼──────────────────
频谱 vs STFT   │ 频谱: 时间平均  │ 频谱: 丢失时变信息
               │ STFT: 时间局部  │ STFT: 时频分辨率折衷
───────────────┼─────────────────┼──────────────────
IQ vs ACF      │ IQ: 瞬时值      │ IQ: 噪声影响大
               │ ACF: 统计量     │ ACF: 短信号估计不准

→ 当某个视图因特定干扰/退化失效时, 其他视图仍可提供判据
→ 这就是多视图的核心价值: 提高鲁棒性
```

```
多视图融合架构：
━━━━━━━━━━━━━━

  IQ [2,8192]     PSD [1,4096]    STFT [1,128,65]    ACF [1,2048]
       │                │                │                 │
       ▼                ▼                ▼                 ▼
  ┌─────────┐    ┌─────────┐     ┌──────────┐      ┌─────────┐
  │1D ResNet│    │1D ResNet│     │2D ResNet │      │1D ResNet│
  │Encoder_1│    │Encoder_2│     │Encoder_3 │      │Encoder_4│
  └────┬────┘    └────┬────┘     └────┬─────┘      └────┬────┘
       │              │               │                  │
       ▼              ▼               ▼                  ▼
    f1∈R^128      f2∈R^128        f3∈R^128          f4∈R^128
       │              │               │                  │
       └──────────────┼───────────────┘──────────────────┘
                      │
                      ▼
              ┌──────────────┐
              │  Attention   │  ← 学习各视图的动态权重
              │  Fusion      │     (某视图质量差时自动降权)
              │              │
              │  α1·f1 ⊕ α2·f2 ⊕ α3·f3 ⊕ α4·f4
              └──────┬───────┘
                     │
                     ▼
               z ∈ R^256  (融合特征)
                     │
              ┌──────┴──────┐
              ▼             ▼
         多任务头...    多任务头...
```

---

## 五、为什么选8192采样点

```
选择 N=8192 的理由（逐条分析）：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 覆盖足够的符号周期
──────────────────────
   Link11数据速率: 1364 bps
   每符号持续时间: 1/1364 ≈ 0.733 ms
   
   采样率 fs = 8000 Hz时:
   8192个采样 = 8192/8000 = 1.024 秒
   
   包含的符号数: 1.024 × 1364 ≈ 1397 个符号
   
   → 约1400个符号, 足够捕获调制的统计特性
   → 足够包含CLEW段或至少一个完整数据块

   对比其他长度:
   ┌──────────┬───────────┬──────────┬──────────────┐
   │ 采样点数  │ 时长(秒)  │ 符号数   │ 够不够?       │
   ├──────────┼───────────┼──────────┼──────────────┤
   │ 1024     │ 0.128     │ ~175     │ ⚠️ 偏少       │
   │ 2048     │ 0.256     │ ~349     │ ⚠️ 勉强       │
   │ 4096     │ 0.512     │ ~699     │ ✅ 可用       │
   │ 8192     │ 1.024     │ ~1397    │ ✅ 推荐 ⭐    │
   │ 16384    │ 2.048     │ ~2793    │ ✅ 充裕但冗余  │
   └──────────┴───────────┴──────────┴──────────────┘


2. 频率分辨率满足需求
──────────────────────
   频率分辨率 Δf = fs / N = 8000 / 8192 ≈ 0.977 Hz
   
   Link11音调间隔: 605 Hz
   → 0.977 Hz分辨率可以精确分辨各音调
   → 每两个音调之间有 605/0.977 ≈ 619 个频率bin
   → 频率估计精度完全足够
   
   对比:
   N=1024 → Δf=7.8Hz (也够, 但精度低)
   N=8192 → Δf≈1Hz   (精确, 可区分相邻信号)


3. 2的幂次 → FFT效率最优
──────────────────────
   8192 = 2^13
   → 所有FFT、STFT运算效率最高
   → GPU上的卷积运算对齐效率高
   → 避免零填充带来的伪影


4. 计算量与信息量的平衡
──────────────────────
   输入维度: [2, 8192] = 16384个数值
   
   · 对于1D-CNN: 完全可处理, 不算大
   · 对于STFT: 窗长256,步长64 → 约[128, 129]的时频图, 适中
   · GPU显存: batch_size=64时, 单个输入约64KB, 可接受
   
   N=16384: 信息增加有限, 计算翻倍, 性价比下降
   N=4096:  可用但信息偏少, 特别是需要频率精度时


5. 匹配Link11帧结构
──────────────────────
   Link11 CLEW持续时间: 约 6.67ms × 重复次数
   典型CLEW段: ~400ms
   典型DATA块: ~数百ms到秒级
   
   1.024秒 ≈ 能覆盖:
   · 一个完整CLEW段 ✅
   · 一段有意义的DATA段 ✅
   · CLEW→DATA过渡 ✅ (如果截取位置合适)


结论: N=8192 在 fs=8000Hz 下是最优平衡点
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

但这不是唯一选择, 实际中可以调整:
  · 如果信号很短/需要快速检测 → N=4096
  · 如果需要更细致的频率分析 → N=16384
  · 如果采样率不是8000Hz → 相应调整使时长≈1秒
```

---

## 六、最终推荐方案：传统+深度混合方案 完整项目结构

### 6.1 精简后的项目目录

```
link11_hybrid_detector/
│
├── configs/
│   ├── default.yaml                  # 所有默认配置集中管理
│   └── experiments/                  # 不同实验配置
│       ├── exp_baseline.yaml
│       └── exp_multi_view.yaml
│
├── data/
│   ├── __init__.py
│   ├── preprocessing.py             # 预处理: 下变频→滤波→归一化→分段
│   ├── multi_view_extractor.py      # 多视图特征提取: IQ/PSD/STFT/ACF
│   ├── augmentation.py              # 数据增强: 噪声/衰落/频偏/音调丢失
│   ├── dataset.py                   # PyTorch Dataset
│   └── datamodule.py                # DataModule (管理train/val/test)
│
├── models/
│   ├── __init__.py
│   ├── encoders/
│   │   ├── __init__.py
│   │   ├── resnet1d.py              # 1D ResNet (处理IQ/PSD/ACF)
│   │   └── resnet2d.py              # 2D ResNet (处理STFT时频图)
│   │
│   ├── fusion.py                    # 多视图注意力融合
│   ├── decoder.py                   # 解码重构头
│   ├── hybrid_model.py             # 完整混合模型 ⭐ (核心)
│   └── losses.py                   # 所有损失函数
│
├── traditional/
│   ├── __init__.py
│   ├── multi_tone_detector.py       # 多音检测
│   ├── clew_matched_filter.py       # CLEW匹配滤波
│   ├── cyclostationary_detector.py  # 循环平稳检测
│   └── traditional_ensemble.py     # 传统方法集成 ⭐
│
├── engine/
│   ├── __init__.py
│   ├── trainer.py                   # 训练循环
│   ├── threshold_calibrator.py      # 阈值校准
│   └── evaluator.py                 # 开放集评估
│
├── inference/
│   ├── __init__.py
│   ├── detector.py                  # 最终检测器 ⭐ (对外接口)
│   └── score_fusion.py             # 深度+传统 分数融合
│
├── utils/
│   ├── __init__.py
│   ├── signal_utils.py              # 信号处理工具函数
│   ├── metrics.py                   # 评估指标
│   ├── visualization.py             # 可视化
│   └── logger.py                    # 日志
│
├── scripts/
│   ├── train.py                     # 训练入口
│   ├── evaluate.py                  # 评估入口
│   ├── calibrate.py                 # 阈值校准入口
│   └── demo_detect.py              # 推理演示
│
└── requirements.txt
```

### 6.2 完整模块连接图

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                     总体架构全景图
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

                  ┌───────────────┐
                  │  Raw IQ File  │
                  │  或实时IQ流    │
                  └───────┬───────┘
                          │
                          ▼
┌──────────────────────────────────────────────────────────┐
│                data/preprocessing.py                      │
│                                                           │
│  raw_iq → 下变频(若需) → 重采样(→8kHz) → 带通滤波     │
│        → 功率归一化 → 固定长度分段(8192) → segments[]   │
└──────────────────────────┬───────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────┐
│            data/multi_view_extractor.py                    │
│                                                           │
│  对每个segment:                                          │
│  ┌─────────────────────────────────────────────────┐     │
│  │ segment (complex, len=8192)                     │     │
│  │    │                                             │     │
│  │    ├──→ view_iq:   [2, 8192]  (I路, Q路)       │     │
│  │    ├──→ view_psd:  [1, 4096]  (welch PSD)      │     │
│  │    ├──→ view_stft: [1, 128, 65] (幅度谱图)     │     │
│  │    └──→ view_acf:  [1, 2048]  (自相关函数)     │     │
│  └─────────────────────────────────────────────────┘     │
│                                                           │
│  输出: MultiViewSample = {                                │
│          'iq': tensor[2,8192],                            │
│          'psd': tensor[1,4096],                           │
│          'stft': tensor[1,128,65],                        │
│          'acf': tensor[1,2048],                           │
│          'label': 0 or 1,                                 │
│          'metadata': {...}                                 │
│        }                                                  │
└──────────────────────────┬───────────────────────────────┘
                           │
            ┌──────────────┴──────────────┐
            │     data/dataset.py          │
            │     data/datamodule.py       │
            │                              │
            │  Link11Dataset(Dataset):     │
            │    __getitem__ → sample      │
            │    训练时调用augmentation     │
            │                              │
            │  Link11DataModule:           │
            │    train_dataloader()        │
            │    val_dataloader()          │
            │    test_dataloader()         │
            └──────────────┬──────────────┘
                           │
━━━━━━━━━━━━━━━━━━━━━━━━━━┼━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
         TRAINING PHASE    │
━━━━━━━━━━━━━━━━━━━━━━━━━━┼━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                           │
                           ▼
┌──────────────────────────────────────────────────────────┐
│               models/hybrid_model.py ⭐                   │
│                                                           │
│  class HybridLink11Model(nn.Module):                     │
│                                                           │
│  ┌─── 多视图编码 ──────────────────────────────────┐    │
│  │                                                   │    │
│  │  iq[2,8192]   psd[1,4096]  stft[1,128,65] acf[1,2048]│
│  │      │             │            │            │    │    │
│  │      ▼             ▼            ▼            ▼    │    │
│  │  ┌────────┐  ┌────────┐  ┌─────────┐  ┌────────┐│    │
│  │  │ResNet  │  │ResNet  │  │ResNet   │  │ResNet  ││    │
│  │  │1D      │  │1D      │  │2D       │  │1D      ││    │
│  │  │enc_iq  │  │enc_psd │  │enc_stft │  │enc_acf ││    │
│  │  └───┬────┘  └───┬────┘  └────┬────┘  └───┬────┘│    │
│  │      │            │            │            │     │    │
│  │      ▼            ▼            ▼            ▼     │    │
│  │   f_iq∈R^d    f_psd∈R^d   f_stft∈R^d   f_acf∈R^d│    │
│  │      │            │            │            │     │    │
│  │      └────────────┼────────────┘────────────┘     │    │
│  │                   ▼                               │    │
│  │          ┌─────────────────┐                      │    │
│  │          │ Attention Fusion│                      │    │
│  │          │                 │                      │    │
│  │          │ α = softmax(    │                      │    │
│  │          │   W·[f1;f2;f3;f4])                    │    │
│  │          │                 │                      │    │
│  │          │ z = Σ αi · fi   │                      │    │
│  │          └────────┬────────┘                      │    │
│  │                   │                               │    │
│  │              z ∈ R^D (融合后的特征向量)            │    │
│  │                                                   │    │
│  └───────────────────┼───────────────────────────────┘    │
│                      │                                     │
│  ┌─── 多任务头 ──────┼────────────────────────────────┐   │
│  │                   │                                 │   │
│  │    ┌──────────────┼─────────────────┐              │   │
│  │    │              │                 │              │   │
│  │    ▼              ▼                 ▼              │   │
│  │ ┌─────────┐  ┌──────────┐  ┌────────────────┐    │   │
│  │ │Decoder  │  │SVDD Head │  │Prototype Head │    │   │
│  │ │         │  │          │  │               │    │   │
│  │ │z→iq_hat │  │c(center) │  │p(prototype)   │    │   │
│  │ │[2,8192] │  │‖z-c‖²   │  │‖z-p‖²        │    │   │
│  │ └────┬────┘  └────┬─────┘  └───────┬───────┘    │   │
│  │      │            │                │             │   │
│  │      ▼            ▼                ▼             │   │
│  │  L_recon      L_svdd          L_proto            │   │
│  │  ‖iq-iq_hat‖² ‖z-c‖²      ‖z-μ_link11‖²     │   │
│  │      │            │                │             │   │
│  │      └────────────┼────────────────┘             │   │
│  │                   ▼                              │   │
│  │         ┌──────────────────┐                     │   │
│  │         │ Combined Loss    │                     │   │
│  │         │ L = λ1·L_recon   │                     │   │
│  │         │   + λ2·L_svdd    │                     │   │
│  │         │   + λ3·L_proto   │                     │   │
│  │         └──────────────────┘                     │   │
│  │                                                  │   │
│  └──────────────────────────────────────────────────┘   │
│                                                          │
└──────────────────────────┬───────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────┐
│                engine/trainer.py                          │
│                                                           │
│  class Trainer:                                          │
│    def train(model, datamodule, config):                 │
│                                                           │
│      # Phase 1: 预训练编码器 (可选, 对比学习)            │
│      pretrain_contrastive(model.encoders, datamodule)    │
│                                                           │
│      # Phase 2: 联合训练                                 │
│      for epoch in range(config.epochs):                  │
│        for batch in datamodule.train_dataloader():       │
│          outputs = model(batch)                          │
│          loss = model.compute_loss(outputs, batch)       │
│          loss.backward()                                 │
│          optimizer.step()                                │
│                                                           │
│        # 验证                                            │
│        val_metrics = evaluate(model, datamodule.val)     │
│        # 早停判断                                        │
│        early_stopping(val_metrics)                       │
│        # 保存checkpoint                                  │
│        save_checkpoint(model, epoch, val_metrics)        │
│                                                           │
│      # Phase 3: 计算SVDD中心和原型                       │
│      model.compute_center(datamodule.train)              │
│      model.compute_prototype(datamodule.train)           │
│                                                           │
└──────────────────────────┬───────────────────────────────┘
                           │
━━━━━━━━━━━━━━━━━━━━━━━━━━┼━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
     CALIBRATION PHASE     │
━━━━━━━━━━━━━━━━━━━━━━━━━━┼━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                           │
                           ▼
┌──────────────────────────────────────────────────────────┐
│            engine/threshold_calibrator.py                  │
│                                                           │
│  输入: 训练好的模型 + 验证集                              │
│                                                           │
│  Step 1: 对验证集所有样本计算分数                         │
│  ┌────────────────────────────────────────────┐          │
│  │ For each val_sample:                       │          │
│  │                                             │          │
│  │   # 深度学习分数 (来自model)                │          │
│  │   dl_scores = model.compute_scores(sample) │          │
│  │     → recon_score: 重构误差                │          │
│  │     → svdd_score: 到超球中心距离           │          │
│  │     → proto_score: 到原型距离              │          │
│  │                                             │          │
│  │   # 传统方法分数 (来自traditional模块)      │          │
│  │   trad_scores = traditional.compute(sample)│          │
│  │     → tone_score: 多音检测分数             │          │
│  │     → clew_score: CLEW匹配分数             │          │
│  │     → cyclo_score: 循环平稳分数            │          │
│  │                                             │          │
│  │   all_scores[sample_id] = {dl + trad}      │          │
│  └────────────────────────────────────────────┘          │
│                                                           │
│  Step 2: 分数分布分析                                     │
│  ┌────────────────────────────────────────────┐          │
│  │                                             │          │
│  │  link11_scores: 所有Link11样本的分数分布   │          │
│  │  non_link11_scores: 非Link11样本的分数分布 │          │
│  │                                             │          │
│  │  分析分离度: 两个分布是否可分              │          │
│  │  可视化: 分数直方图, 各维度散点图          │          │
│  └────────────────────────────────────────────┘          │
│                                                           │
│  Step 3: 确定融合权重和阈值                               │
│  ┌────────────────────────────────────────────┐          │
│  │                                             │          │
│  │  # 方法1: 对各分数做min-max归一化          │          │
│  │  norm_scores = normalize(all_scores)        │          │
│  │                                             │          │
│  │  # 方法2: 网格搜索最优权重                  │          │
│  │  best_weights = grid_search(                │          │
│  │    norm_scores, labels,                     │          │
│  │    objective=maximize_auc                   │          │
│  │  )                                          │          │
│  │  → w = {w_recon, w_svdd, w_proto,          │          │
│  │         w_tone, w_clew, w_cyclo}            │          │
│  │                                             │          │
│  │  # 方法3: 在融合分数上确定阈值              │          │
│  │  fused = weighted_sum(norm_scores, w)       │          │
│  │  threshold = find_threshold(                │          │
│  │    fused, labels,                           │          │
│  │    target_tpr=0.99  # 99%检测率             │          │
│  │  )                                          │          │
│  │                                             │          │
│  └────────────────────────────────────────────┘          │
│                                                           │
│  输出: CalibrationResult = {                              │
│    'score_normalizers': {...},  # 各分数的归一化参数      │
│    'fusion_weights': {...},     # 融合权重                │
│    'threshold': float,          # 决策阈值                │
│    'performance': {...}         # 校准集上的性能          │
│  }                                                        │
│                                                           │
└──────────────────────────┬───────────────────────────────┘
                           │
━━━━━━━━━━━━━━━━━━━━━━━━━━┼━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
     INFERENCE PHASE       │
━━━━━━━━━━━━━━━━━━━━━━━━━━┼━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                           │
                           ▼
┌──────────────────────────────────────────────────────────┐
│              inference/detector.py ⭐                      │
│              (对外的唯一接口)                               │
│                                                           │
│  class Link11Detector:                                   │
│                                                           │
│    组件:                                                  │
│    ├── preprocessor    ← data/preprocessing.py           │
│    ├── view_extractor  ← data/multi_view_extractor.py    │
│    ├── dl_model        ← models/hybrid_model.py (frozen) │
│    ├── trad_detector   ← traditional/traditional_ensemble│
│    ├── score_fusion    ← inference/score_fusion.py       │
│    └── calibration     ← 加载校准结果                     │
│                                                           │
│    调用流程:                                              │
│    ┌──────────────────────────────────────────────┐      │
│    │                                              │      │
│    │  def detect(raw_signal) → DetectionResult:   │      │
│    │                                              │      │
│    │  ①  segments = preprocessor(raw_signal)      │      │
│    │         │                                    │      │
│    │  ②  views = view_extractor(segments)         │      │
│    │         │                                    │      │
│    │  ③  ┌───┴───────────────────────────┐       │      │
│    │     │                               │       │      │
│    │     ▼                               ▼       │      │
│    │  dl_scores = dl_model.            trad_scores│      │
│    │    compute_scores(views)   = trad_detector.  │      │
│    │     │                       compute(segments)│      │
│    │     │  · recon_score              │          │      │
│    │     │  · svdd_score        · tone_score     │      │
│    │     │  · proto_score       · clew_score     │      │
│    │     │                      · cyclo_score    │      │
│    │     └───────────┬───────────────────┘       │      │
│    │                 │                            │      │
│    │  ④  fused_score = score_fusion(              │      │
│    │       dl_scores, trad_scores,                │      │
│    │       weights, normalizers                   │      │
│    │     )                                        │      │
│    │                 │                            │      │
│    │  ⑤  is_link11 = fused_score > threshold     │      │
│    │     confidence = sigmoid(margin/temperature) │      │
│    │                 │                            │      │
│    │  ⑥  return DetectionResult(                  │      │
│    │       is_link11,                             │      │
│    │       confidence,                            │      │
│    │       all_scores,                            │      │
│    │       explanation                            │      │
│    │     )                                        │      │
│    │                                              │      │
│    └──────────────────────────────────────────────┘      │
│                                                           │
└──────────────────────────────────────────────────────────┘
```

### 6.3 各模块的接口定义

```python
# ================================================================
# 各模块之间的接口约定 (Interface Contracts)
# ================================================================

# ---- 数据结构定义 ----

@dataclass
class RawSignal:
    """原始信号"""
    iq_data: np.ndarray          # complex64, shape: [N]
    sample_rate: float           # Hz
    metadata: dict               # 元信息

@dataclass 
class MultiViewSample:
    """多视图样本 (Dataset输出 / Model输入)"""
    iq: torch.Tensor             # [2, 8192]
    psd: torch.Tensor            # [1, 4096]
    stft: torch.Tensor           # [1, 128, 65]
    acf: torch.Tensor            # [1, 2048]
    label: int                   # 1=link11, 0=non_link11
    metadata: dict

@dataclass
class ModelOutput:
    """模型前向输出"""
    latent: torch.Tensor         # [B, D] 融合特征
    reconstruction: torch.Tensor # [B, 2, 8192] 重构IQ
    svdd_distance: torch.Tensor  # [B] 到中心距离
    proto_distance: torch.Tensor # [B] 到原型距离
    view_features: dict          # 各视图的独立特征
    attention_weights: torch.Tensor  # [B, 4] 各视图权重

@dataclass
class AnomalyScores:
    """异常分数集合"""
    # 深度学习分数 (越小越像Link11)
    recon_score: float           # 重构误差
    svdd_score: float            # SVDD距离
    proto_score: float           # 原型距离
    # 传统方法分数 (越大越像Link11)  
    tone_score: float            # 多音检测分数
    clew_score: float            # CLEW匹配分数
    cyclo_score: float           # 循环平稳分数

@dataclass
class DetectionResult:
    """最终检测结果"""
    is_link11: bool              # 是否为Link11
    confidence: float            # 置信度 [0, 1]
    fused_score: float           # 融合后分数
    individual_scores: AnomalyScores  # 各项分数
    explanation: str             # 可读解释
    processing_time_ms: float    # 处理耗时

@dataclass
class CalibrationConfig:
    """校准配置 (序列化保存)"""
    score_normalizers: dict      # 各分数的(min, max)
    fusion_weights: dict         # 各分数的融合权重
    threshold: float             # 决策阈值
    temperature: float           # 置信度温度参数


# ---- 模块接口 ----

# data/preprocessing.py
class Preprocessor:
    def __call__(self, raw: RawSignal) -> List[np.ndarray]:
        """原始信号 → 预处理后的segments列表"""
        ...

# data/multi_view_extractor.py  
class MultiViewExtractor:
    def __call__(self, segment: np.ndarray) -> MultiViewSample:
        """单个segment → 多视图特征"""
        ...

# data/augmentation.py
class SignalAugmentor:
    def __call__(self, segment: np.ndarray) -> np.ndarray:
        """信号增强 (仅训练时)"""
        ...

# models/hybrid_model.py
class HybridLink11Model(nn.Module):
    def forward(self, sample: MultiViewSample) -> ModelOutput:
        """前向传播"""
        ...
    def compute_loss(self, output: ModelOutput, target) -> torch.Tensor:
        """计算组合损失"""
        ...
    def compute_scores(self, sample: MultiViewSample) -> AnomalyScores:
        """推理时计算异常分数 (no_grad)"""
        ...

# traditional/traditional_ensemble.py
class TraditionalEnsemble:
    def compute_scores(self, segment: np.ndarray, fs: float) -> dict:
        """传统方法分数"""
        ...

# engine/trainer.py
class Trainer:
    def train(self, model, datamodule, config) -> None:
        """完整训练流程"""
        ...

# engine/threshold_calibrator.py
class ThresholdCalibrator:
    def calibrate(self, model, trad_detector, val_data) -> CalibrationConfig:
        """校准阈值和权重"""
        ...

# inference/detector.py
class Link11Detector:
    def detect(self, raw_signal: RawSignal) -> DetectionResult:
        """端到端检测"""
        ...
    def detect_batch(self, signals: List[RawSignal]) -> List[DetectionResult]:
        """批量检测"""
        ...
```

### 6.4 配置文件模板

```yaml
# configs/default.yaml

# ============ 数据配置 ============
data:
  sample_rate: 8000           # Hz
  segment_length: 8192        # 采样点
  segment_hop: 4096           # 滑动步长(50%重叠)
  bandpass_low: 300           # Hz
  bandpass_high: 3400         # Hz
  
  # 多视图参数
  views:
    iq:
      enabled: true
      shape: [2, 8192]
    psd:
      enabled: true
      nfft: 8192
      output_length: 4096     # 只取正频率
    stft:
      enabled: true
      win_length: 256
      hop_length: 64
      n_fft: 128
      # 输出: [1, T≈128, F=65]
    acf:
      enabled: true
      max_lag: 2048
  
  # 数据增强
  augmentation:
    enabled: true             # 仅训练时
    awgn_snr_range: [-5, 30]  # dB
    freq_offset_range: [-50, 50]  # Hz
    phase_noise_std: 0.1      # rad
    multipath_enabled: true
    tone_drop_prob: 0.3       # 随机丢失音调的概率
    tone_drop_max: 10         # 最多丢失几个音调
    time_shift_range: [-200, 200]  # 采样点

  # 数据集路径
  dataset:
    train_link11_dir: "data/train/link11/"
    train_non_link11_dir: "data/train/non_link11/"
    val_dir: "data/val/"
    test_dir: "data/test/"

# ============ 模型配置 ============
model:
  # 各视图编码器
  encoders:
    iq:
      type: "resnet1d"
      in_channels: 2
      base_channels: 64
      num_blocks: [2, 2, 2, 2]  # ResNet-18 style
      output_dim: 128
    psd:
      type: "resnet1d"
      in_channels: 1
      base_channels: 32
      num_blocks: [2, 2, 2]
      output_dim: 128
    stft:
      type: "resnet2d"
      in_channels: 1
      base_channels: 32
      num_blocks: [2, 2, 2]
      output_dim: 128
    acf:
      type: "resnet1d"
      in_channels: 1
      base_channels: 32
      num_blocks: [2, 2]
      output_dim: 128
  
  # 融合
  fusion:
    type: "attention"         # "concat" | "attention" | "gated"
    input_dim: 128            # 每个视图的特征维度
    num_views: 4
    fused_dim: 256            # 融合后维度
  
  # 解码器(重构头)
  decoder:
    output_shape: [2, 8192]   # 重构IQ
    hidden_dims: [256, 512, 1024]
  
  # SVDD
  svdd:
    enabled: true
  
  # 原型
  prototype:
    enabled: true
    update_momentum: 0.99     # EMA更新原型

# ============ 损失配置 ============
loss:
  lambda_recon: 1.0           # 重构损失权重
  lambda_svdd: 0.1            # SVDD损失权重  
  lambda_proto: 0.1           # 原型损失权重
  svdd_nu: 0.05              # SVDD软边界参数

# ============ 训练配置 ============
training:
  epochs: 200
  batch_size: 64
  optimizer:
    type: "adamw"
    lr: 1.0e-3
    weight_decay: 1.0e-5
  scheduler:
    type: "cosine"
    T_max: 200
    eta_min: 1.0e-6
  early_stopping:
    patience: 20
    metric: "val_auc"
  
  # 预训练阶段(可选)
  pretrain:
    enabled: true
    epochs: 50
    method: "simclr"          # 对比学习预训练
    temperature: 0.1

# ============ 传统方法配置 ============
traditional:
  multi_tone:
    enabled: true
    expected_spacing: 605     # Hz
    spacing_tolerance: 50     # Hz
    min_tones: 5
    peak_snr_threshold: 6    # dB above noise floor
  clew_matched_filter:
    enabled: true
    template_duration: 0.1    # 秒
  cyclostationary:
    enabled: true
    cyclic_freqs: [605, 1210, 1364]  # 待检测的循环频率

# ============ 校准配置 ============
calibration:
  target_tpr: 0.99            # 目标检测率
  method: "roc_optimal"       # "percentile" | "roc_optimal" | "evt"
  fusion_weight_search: "grid" # "grid" | "bayesian"

# ============ 推理配置 ============
inference:
  device: "cuda"
  batch_size: 128
  confidence_temperature: 1.0
```

### 6.5 训练到推理的完整执行流程

```
完整执行流程 (从零到可用):
━━━━━━━━━━━━━━━━━━━━━━━

Step 0: 数据准备
──────────────────
$ python scripts/prepare_data.py --config configs/default.yaml
  │
  ├── 读取原始IQ文件
  ├── 预处理 + 分段
  ├── 提取多视图特征
  ├── 保存为.npy / .pt格式
  ├── 生成metadata.csv
  └── 划分train/val/test

Step 1: 训练
──────────────────
$ python scripts/train.py --config configs/default.yaml
  │
  ├── Phase 1 (可选): 对比学习预训练编码器
  │   └── 50 epochs, 仅Link11数据, SimCLR
  │
  ├── Phase 2: 联合训练
  │   ├── 加载预训练编码器权重
  │   ├── 200 epochs, 多任务损失
  │   ├── 验证集监控 + 早停
  │   └── 保存best checkpoint
  │
  └── Phase 3: 后处理
      ├── 加载best model
      ├── 遍历训练集计算SVDD中心 c = mean(z_train)
      └── 遍历训练集计算原型 p = mean(z_link11)
      保存→ model_final.pt

Step 2: 校准
──────────────────
$ python scripts/calibrate.py --model model_final.pt --config configs/default.yaml
  │
  ├── 对验证集计算所有分数 (深度+传统)
  ├── 分析分数分布
  ├── 搜索最优融合权重
  ├── 确定决策阈值 (target TPR=99%)
  ├── 输出性能报告
  └── 保存→ calibration.yaml

Step 3: 评估
──────────────────
$ python scripts/evaluate.py --model model_final.pt --calib calibration.yaml
  │
  ├── 在测试集上运行检测器
  ├── 计算指标:
  │   ├── TPR (True Positive Rate): 正确检测Link11
  │   ├── FPR (False Positive Rate): 误判非Link11为Link11
  │   ├── AUROC: ROC曲线下面积
  │   ├── F1-Score
  │   └── 不同SNR下的TPR曲线
  │
  ├── 开放集评估:
  │   ├── 已知非目标类的拒绝率
  │   └── 未知非目标类的拒绝率 (关键!)
  │
  └── 输出评估报告 + 可视化

Step 4: 部署推理
──────────────────
# 使用方式:
from inference.detector import Link11Detector

detector = Link11Detector.load(
    model_path="model_final.pt",
    calibration_path="calibration.yaml",
    config_path="configs/default.yaml"
)

# 检测单个信号
result = detector.detect(raw_iq_signal)
print(f"Is Link11: {result.is_link11}")
print(f"Confidence: {result.confidence:.3f}")
print(f"Scores: {result.individual_scores}")
print(f"Explanation: {result.explanation}")

# 示例输出:
# Is Link11: True
# Confidence: 0.973
# Scores: AnomalyScores(recon=0.023, svdd=0.15, proto=0.12, 
#                        tone=0.92, clew=0.87, cyclo=0.78)
# Explanation: "高置信度Link11信号. 检测到12个等间隔音调(间隔603Hz),
#              CLEW匹配分数0.87, 重构误差低(0.023)."
```

### 6.6 关键设计决策总结

```
┌──────────────────────────────────────────────────────────┐
│                   设计决策总结                             │
│                                                           │
│  1. 为什么多视图而非单一输入?                            │
│     → 鲁棒性: 某一视图退化时其他视图补偿                 │
│     → 互补性: 时域/频域/时频域提供不同角度               │
│     → 注意力机制自动学习各视图的可信度                   │
│                                                           │
│  2. 为什么同时用重构+SVDD+原型?                          │
│     → 重构: 建模Link11的生成过程                         │
│     → SVDD: 约束特征空间紧凑                            │
│     → 原型: 提供直观的距离度量                           │
│     → 三者失效模式不同, 互相保险                         │
│                                                           │
│  3. 为什么融合传统方法?                                  │
│     → Link11有明确物理结构(多音/CLEW), 传统方法直接利用  │
│     → 传统方法不需要训练, 零样本即可工作                 │
│     → 深度方法在退化条件下补充传统方法的不足             │
│     → 融合后比任何单一方法都鲁棒                         │
│                                                           │
│  4. 为什么阈值需要单独校准?                              │
│     → 训练时的loss不直接等于最优决策边界                 │
│     → 需要在验证集上选择满足业务需求(如TPR≥99%)的阈值   │
│     → 不同部署场景可能需要不同的阈值(误报容忍度不同)     │
│                                                           │
│  5. 为什么开放集评估要测试"未知"类?                      │
│     → 这是项目的核心挑战                                 │
│     → 模型必须对训练中从未见过的信号也能正确拒绝         │
│     → 只在"已知"非目标上评估会高估实际性能               │
│                                                           │
└──────────────────────────────────────────────────────────┘
```